# 02 — Train GLHMM

Fit a Gaussian Linear HMM on the observation matrix `Y` loaded in notebook 01, then save the model pickle and training metadata.

Requires **glhmm**. If it is missing and `skip_train_if_no_glhmm` is false, this notebook raises an error.

In [1]:
import json
import os
import pickle
from pathlib import Path

import numpy as np
import yaml

PROJECT_ROOT = Path('..').resolve()
CONFIG_NAME = os.environ.get('BRAIN_STATES_CONFIG', 'config.yaml')
with open(PROJECT_ROOT / CONFIG_NAME, encoding='utf-8') as f:
    cfg = yaml.safe_load(f)

paths = cfg['paths']
DATA_DIR = (PROJECT_ROOT / paths['data_dir']).resolve()
HMM_DIR = (PROJECT_ROOT / paths['hmm_models_dir']).resolve()
RESULTS_DIR = (PROJECT_ROOT / paths['results_dir']).resolve()
HMM_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

with open(RESULTS_DIR / 'analysis_metadata.json', encoding='utf-8') as f:
    meta = json.load(f)

use_example_data = bool(meta['use_example_data'])
n_parcels = int(meta['n_parcels'])
K = int(meta['K'])
method = meta['method']
random_seed = int(meta['random_seed'])
model_beta = meta.get('model_beta', 'no')
covtype = meta.get('covtype', 'full')
model_mean = meta.get('model_mean', 'state')
skip_if_missing = bool(meta.get('skip_train_if_no_glhmm', False))

Y = np.load(RESULTS_DIR / 'Y.npy')
indices = np.load(RESULTS_DIR / 'indices.npy')
print(f'Loaded Y {Y.shape}, indices {indices.shape}')


Loaded Y (360, 18), indices (3, 2)


In [2]:
try:
    from glhmm import glhmm as glhmm_mod
    has_glhmm = True
except ImportError:
    has_glhmm = False
    glhmm_mod = None

model_name = f'trained_hmm_{n_parcels}parcels_{K}states_{method}.pkl'
model_path = HMM_DIR / model_name
train_info = {
    'trained': False,
    'skipped': False,
    'model_path': (model_path.relative_to(PROJECT_ROOT)).as_posix(),
    'model_name': model_name,
    'K': K,
    'n_parcels': n_parcels,
    'method': method,
    'random_seed': random_seed,
    'model_beta': model_beta,
    'covtype': covtype,
    'model_mean': model_mean,
    'Y_shape': list(Y.shape),
    'indices_shape': list(indices.shape),
}

if not has_glhmm:
    msg = (
        'glhmm is not installed.'
    )
    if use_example_data and skip_if_missing:
        print('Example mode: ' + msg)
        print('Skipping HMM training (skip_train_if_no_glhmm=true).')
        train_info['skipped'] = True
        train_info['skip_reason'] = 'glhmm_not_installed'
    else:
        raise ImportError(msg)
else:
    np.random.seed(random_seed)
    hmm = glhmm_mod.glhmm(
        model_beta=model_beta,
        K=K,
        covtype=covtype,
        model_mean=model_mean,
    )
    print(f'Fitting GLHMM: K={K}, covtype={covtype}, model_mean={model_mean}')
    Gamma, Xi, FE = hmm.train(Y=Y, indices=indices, X=None)
    K_fit = int(hmm.hyperparameters['K'])
    with open(model_path, 'wb') as f:
        pickle.dump(hmm, f, protocol=pickle.HIGHEST_PROTOCOL)
    # Round-trip load to confirm the artifact is readable
    with open(model_path, 'rb') as f:
        hmm = pickle.load(f)

    train_info.update({
        'trained': True,
        'K_fit': K_fit,
        'Gamma_shape': list(np.shape(Gamma)),
        'free_energy': float(np.asarray(FE).reshape(-1)[-1]) if np.size(FE) else None,
    })
    print(f'Saved model: {model_path}')
    print(f'Gamma shape: {np.shape(Gamma)}')


Fitting GLHMM: K=3, covtype=full, model_mean=state
Init repetition 1 free energy = 6299.727732549094
Init repetition 2 free energy = 6343.0285836563735


Init repetition 3 free energy = 6308.135879403389
Init repetition 4 free energy = 6348.206451356797
Init repetition 5 free energy = 6336.077204904879
Best repetition: 1
Cycle 1 free energy = 6302.421284724938
Cycle 2 free energy = 6298.165779274486
Cycle 3, free energy = 6295.859938053231, relative change = 0.35142804314820636
Cycle 4, free energy = 6293.635023828986, relative change = 0.2532265147362494
Cycle 5, free energy = 6290.626791853946, relative change = 0.25505394830824224
Cycle 6, free energy = 6290.225463970005, relative change = 0.03290700084935099
Cycle 7, free energy = 6290.122695213107, relative change = 0.008356141718425763


Cycle 8, free energy = 6289.935276854644, relative change = 0.015010270729454926
Cycle 9, free energy = 6289.3351777800635, relative change = 0.04585772354669978
Cycle 10, free energy = 6287.860719186181, relative change = 0.1012638272845755
Cycle 11, free energy = 6286.378892716337, relative change = 0.09236942153325586
Cycle 12, free energy = 6285.678614444318, relative change = 0.04182596087015863
Cycle 13, free energy = 6284.90795776747, relative change = 0.04400401355602178
Cycle 14, free energy = 6283.253137924117, relative change = 0.08633175969218103
Cycle 15, free energy = 6282.17836628811, relative change = 0.05309370975145905
Cycle 16, free energy = 6281.630449401282, relative change = 0.026353769740284378
Cycle 17, free energy = 6280.9586941820735, relative change = 0.03129888807536612
Cycle 18, free energy = 6280.621748270232, relative change = 0.01545656314946433
Cycle 19, free energy = 6280.462725131727, relative change = 0.007241965841623518
Cycle 20, free energy = 6280

Cycle 36, free energy = 6278.690753680494, relative change = 0.00011619172046069658
Cycle 37, free energy = 6278.68746287024, relative change = 0.00013865488137883156
Cycle 38, free energy = 6278.683231783816, relative change = 0.00017824066844107262
Cycle 39, free energy = 6278.677308199154, relative change = 0.0002494773634728576
Cycle 40, free energy = 6278.668162978139, relative change = 0.00038501133084200786
Cycle 41, free energy = 6278.652404441906, relative change = 0.0006629902648097589
Cycle 42, free energy = 6278.622020740594, relative change = 0.0012766655864487268
Cycle 43, free energy = 6278.558674283031, relative change = 0.002654632347004609
Cycle 44, free energy = 6278.434201078436, relative change = 0.005189176242914488
Cycle 45, free energy = 6278.266821194374, relative change = 0.006929563302046873
Cycle 46, free energy = 6278.162064355139, relative change = 0.004318227776415425
Cycle 47, free energy = 6278.131855724821, relative change = 0.0012436945437087894
Cycle

In [3]:
train_meta_path = RESULTS_DIR / 'training_metadata.json'
with open(train_meta_path, 'w', encoding='utf-8') as f:
    json.dump(train_info, f, ensure_ascii=False, indent=2)
print(f'Saved training metadata: {train_meta_path}')
if train_info.get('trained'):
    print('Done.')
elif train_info.get('skipped'):
    print('Done (training skipped in example mode).')
else:
    print('Done.')


Saved training metadata: D:\文件同步\文件\数据分析结果\data_code_figure\adaptive_reserve_moderate\brain_states\results\example\training_metadata.json
Done.
